# Step 26 — Inference with test-time augmentation

**Settings: `GPU T4 x2`, `Internet OFF`.**

**Inputs:** competition data, `rsna-knee-checkpoints`, `rsna-knee-resnet-ckpts`

## What TTA does

Predict each study several times under small geometric variations, then average. A model's errors depend partly on incidental framing — exactly where the joint sits in the frame, at what angle. Averaging over perturbations cancels some of that, the same way ensembling over models cancels weight-dependent error.

It requires no training. You already have the ten models.

## Three views

| view | rotation | scale |
|---|---|---|
| 1 | 0 | 1.00 |
| 2 | +7 | 1.03 |
| 3 | -7 | 0.97 |

Deliberately **milder than training augmentation** (which used ±12 degrees and 0.9–1.1). TTA views should sit near the centre of the training distribution, not at its edges — you want plausible variations the model handles well, not hard cases.

**No flips.** Every knee was canonicalised to a right-knee orientation, so mirroring would swap medial and lateral and corrupt four labels.

## Runtime

Decoding happens once and is shared across all views; only the forward passes repeat.

| | minutes |
|---|---|
| no TTA (current) | 141 |
| **TTA x3** | **395** |
| TTA x4 | 522 — too close to the 540 limit |

x3 leaves ~145 minutes of margin. If it ever runs long, drop `TTA_VIEWS` to two entries.

The affine transforms run on the GPU via `grid_sample` rather than on CPU — at 124,800 slices per view, CPU warping would dominate the runtime.

### 1 · Setup

In [ ]:
import os, gc, glob, time, math, zipfile, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, cv2, pydicom, torch, torch.nn as nn
import torch.nn.functional as F
from multiprocessing import Pool

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
PX, NSL, NSER = 224, 32, 3
LAT_THR, LAT_SIGN = -37.5, 1
W_B0 = 0.50

# (rotation degrees, scale) -- first entry is the identity view
TTA_VIEWS = [(0.0, 1.00), (7.0, 1.03), (-7.0, 0.97)]

LABELS = pd.read_csv(f"{ROOT}/sample_submission.csv").columns[1:].tolist()
test = pd.read_csv(f"{ROOT}/test.csv")
tser = pd.read_csv(f"{ROOT}/test_series.csv")
studies = test["StudyInstanceUID"].tolist()
print(f"test studies: {len(studies):,} | TTA views: {len(TTA_VIEWS)} | w_b0 {W_B0}")

### 2 · Rebuild and load both model families

In [ ]:
roots = [p for p in glob.glob("/kaggle/input/*") if not p.endswith("competitions")]
pkls = []
for r in roots:
    pkls += glob.glob(f"{r}/**/data.pkl", recursive=True)

def rebuild(src, out):
    with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
        for r, d, files in os.walk(src):
            for n in files:
                full = os.path.join(r, n)
                rel  = os.path.relpath(full, src).replace("\\", "/")
                z.write(full, "archive/" + rel)
    return out

CKPTS = []
for i, pkl in enumerate(sorted(pkls)):
    src = os.path.dirname(pkl)
    tags = [p for p in src.split("/") if "fold" in p.lower()]
    CKPTS.append(rebuild(src, f"/kaggle/working/{(tags[-1] if tags else f'ck{i}')}.pt"))
print(f"rebuilt {len(CKPTS)} checkpoints")

In [ ]:
import timm

class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Sequential(nn.Linear(d, d // 2), nn.Tanh(), nn.Linear(d // 2, 1))
    def forward(self, x, mask=None):
        a = self.w(x).squeeze(-1)
        if mask is not None:
            a = a.masked_fill(mask < 0.5, -1e4)
        return (x * a.softmax(-1).unsqueeze(-1)).sum(1)

class Net(nn.Module):
    def __init__(self, backbone, nlab=12, hid=256):
        super().__init__()
        self.enc = timm.create_model(backbone, pretrained=False, num_classes=0, in_chans=3)
        d = self.enc.num_features
        self.gru        = nn.GRU(d, hid, batch_first=True, bidirectional=True)
        self.slice_pool = AttnPool(hid * 2)
        self.ser_pool   = AttnPool(hid * 2)
        self.head       = nn.Sequential(nn.Dropout(0.3), nn.Linear(hid * 2, nlab))
    def forward(self, v, mask):
        B, S, T, H, W = v.shape
        lo = torch.cat([v[:, :, :1], v[:, :, :-1]], 2)
        hi = torch.cat([v[:, :, 1:], v[:, :, -1:]], 2)
        x  = torch.stack([lo, v, hi], 3).reshape(B * S * T, 3, H, W)
        f = self.enc(x).reshape(B * S, T, -1)
        f, _ = self.gru(f)
        f = self.slice_pool(f).reshape(B, S, -1)
        return self.head(self.ser_pool(f, mask))

M_B0, M_RN = [], []
for p in CKPTS:
    ck = torch.load(p, map_location="cpu", weights_only=False)
    bb = ck.get("backbone", "efficientnet_b0")
    m = Net(bb).cuda(); m.load_state_dict(ck["model"]); m.eval()
    (M_RN if "resnet" in bb else M_B0).append(m)

print(f"EfficientNet-B0: {len(M_B0)}   ResNet34d: {len(M_RN)}")
assert M_B0 and M_RN, "a model family is missing"

### 3 · GPU affine for the TTA views

`affine_grid` + `grid_sample` rotate and scale about the image centre. Running this on the GPU matters: at 124,800 slices per view, CPU warping would cost more than the forward passes.

In [ ]:
def tta_transform(v, angle_deg, scale):
    """v: (B, S, T, H, W) on GPU. Rotate/scale every slice about its centre."""
    if angle_deg == 0.0 and scale == 1.0:
        return v
    B, S, T, H, W = v.shape
    x = v.reshape(B * S * T, 1, H, W)
    a = math.radians(angle_deg)
    cos, sin = math.cos(a) / scale, math.sin(a) / scale
    theta = torch.tensor([[cos, -sin, 0.0], [sin, cos, 0.0]],
                         dtype=x.dtype, device=x.device).unsqueeze(0).expand(x.size(0), -1, -1)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    out = F.grid_sample(x, grid, align_corners=False, padding_mode="zeros")
    return out.reshape(B, S, T, H, W)

# sanity: identity must be a no-op, rotation must actually change something
_t = torch.rand(1, 1, 2, 16, 16).cuda()
print("identity unchanged:", torch.allclose(tta_transform(_t, 0.0, 1.0), _t))
print("rotation changes  :", not torch.allclose(tta_transform(_t, 7.0, 1.03), _t))
del _t; torch.cuda.empty_cache()

### 4 · Decoder — unchanged from the 224px cache

In [ ]:
def pick(g):
    out = []
    for plane in ["Sagittal", "Coronal", "Axial"]:
        sub = g[g["Anatomical_Plane"] == plane]
        if len(sub) == 0:
            out.append(None); continue
        sub = sub.sort_values(["Fluid_Sensitive", "Fat_Suppression"], ascending=False)
        out.append(sub.iloc[0]["SeriesInstanceUID"])
    return out

chosen = {suid: pick(g) for suid, g in tser.groupby("StudyInstanceUID")}


def load_series(dirpath):
    files = os.listdir(dirpath)
    if not files: return None
    metas = []
    for f in files:
        try:
            d = pydicom.dcmread(os.path.join(dirpath, f), stop_before_pixels=True)
            iop = getattr(d, "ImageOrientationPatient", None)
            ipp = getattr(d, "ImagePositionPatient", None)
            metas.append((f, np.array(iop, float) if iop else None,
                             np.array(ipp, float) if ipp else None,
                             getattr(d, "InstanceNumber", 0),
                             str(getattr(d, "Laterality", "") or "")))
        except Exception:
            continue
    if not metas: return None

    iops = [m[1] for m in metas if m[1] is not None]
    if iops:
        iop = iops[0]; r, c = iop[:3], iop[3:]; n = np.cross(r, c)
        pos = [(float(np.dot(m[2], n)) if m[2] is not None else float(m[3])) for m in metas]
    else:
        r, c, n = np.array([1.,0,0]), np.array([0,1.,0]), np.array([0,0,1.])
        pos = [float(m[3]) for m in metas]

    order = np.argsort(pos); metas = [metas[i] for i in order]
    take = np.linspace(0, len(metas)-1, NSL).round().astype(int)

    imgs = []
    for i in take:
        try:
            d = pydicom.dcmread(os.path.join(dirpath, metas[i][0]))
            a = d.pixel_array.astype(np.float32)
            if hasattr(d, "RescaleSlope"):
                a = a * float(d.RescaleSlope) + float(getattr(d, "RescaleIntercept", 0))
        except Exception:
            a = np.zeros((PX, PX), np.float32)
        imgs.append(cv2.resize(a, (PX, PX), interpolation=cv2.INTER_AREA))
    vol = np.stack(imgs)

    lo, hi = np.percentile(vol, [1, 99])
    if hi <= lo: hi = lo + 1
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    axes = [n, c, r]
    for ax, vec in enumerate(axes):
        dom = int(np.argmax(np.abs(vec)))
        if vec[dom] < 0: vol = np.flip(vol, axis=ax)
        axes[ax] = vec * (1 if vec[dom] >= 0 else -1)

    tag = next((m[4] for m in metas if m[4]), "")
    if tag:
        side = tag.strip().upper()[0]
    else:
        xs = [float(m[2][0]) for m in metas if m[2] is not None]
        side = ("L" if (np.mean(xs) - LAT_THR) * LAT_SIGN > 0 else "R") if xs else "R"
    if side == "L":
        vol = np.flip(vol, axis=int(np.argmax([abs(v[0]) for v in axes])))

    return (vol * 255).astype(np.uint8)


def load_study(i):
    suid = studies[i]
    out  = np.zeros((NSER, NSL, PX, PX), np.uint8)
    pres = np.zeros(NSER, np.float32)
    for k, cs in enumerate(chosen.get(suid, [None]*3)):
        if cs is None: continue
        p = f"{ROOT}/test_series/{suid}/{cs}"
        if not os.path.isdir(p): continue
        try:
            v = load_series(p)
            if v is not None: out[k] = v; pres[k] = 1.0
        except Exception:
            pass
    return i, out, pres

print("decoder ready")

### 5 · Predict over all views

In [ ]:
CHUNK, BS = 64, 3
preds = np.full((len(studies), 12), 0.5, np.float32)

t0 = time.time()
for c0 in range(0, len(studies), CHUNK):
    idx = list(range(c0, min(c0 + CHUNK, len(studies))))
    with Pool(4) as pool:
        res = pool.map(load_study, idx)

    vols = np.stack([r[1] for r in res]).astype(np.float32) / 255.0
    mask = np.stack([r[2] for r in res])
    ids  = [r[0] for r in res]

    with torch.no_grad():
        for b0 in range(0, len(ids), BS):
            vb_raw = torch.from_numpy(vols[b0:b0+BS]).cuda()
            mb     = torch.from_numpy(mask[b0:b0+BS]).cuda()

            acc_b0 = acc_rn = None
            for ang, sc in TTA_VIEWS:
                vb = tta_transform(vb_raw, ang, sc)
                for m in M_B0:
                    with torch.amp.autocast("cuda"):
                        o = m(vb, mb).float().sigmoid()
                    acc_b0 = o if acc_b0 is None else acc_b0 + o
                for m in M_RN:
                    with torch.amp.autocast("cuda"):
                        o = m(vb, mb).float().sigmoid()
                    acc_rn = o if acc_rn is None else acc_rn + o

            n = len(TTA_VIEWS)
            p_b0 = (acc_b0 / (len(M_B0) * n)).cpu().numpy()
            p_rn = (acc_rn / (len(M_RN) * n)).cpu().numpy()
            preds[ids[b0:b0+BS]] = W_B0 * p_b0 + (1 - W_B0) * p_rn

    del vols; gc.collect()
    done = c0 + len(idx); el = time.time() - t0
    eta = (len(studies) - done) * el / done / 60
    print(f"  {done}/{len(studies)}  {el/done:.2f}s/study  eta {eta:.0f} min", flush=True)
    if el / 60 > 400:
        print("  *** over 400 min, consider dropping a TTA view next time ***", flush=True)

print(f"\ndone in {(time.time()-t0)/60:.1f} min")

### 6 · Write submission

In [ ]:
sub = pd.DataFrame(preds, columns=LABELS)
sub.insert(0, "StudyInstanceUID", studies)
sub.to_csv("submission.csv", index=False)
print(sub.shape)
print(f"rows left at default 0.5: {(preds == 0.5).all(1).sum()}")

for p in CKPTS:
    try: os.remove(p)
    except Exception: pass
print("cleaned up rebuilt checkpoints")